# 02 — Estrategias de recuperación y RAG agéntico

Este notebook demuestra las estrategias de recuperación disponibles en el sistema GraphRAG
sobre los relatos de Sherlock Holmes y muestra cómo la capa agéntica selecciona automáticamente
la más adecuada.

| Estrategia | Cuándo usarla | Cómo funciona |
|---|---|---|
| **Vector search** | Consultas semánticas / conceptuales | Similitud coseno sobre embeddings de chunks |
| **Full-text search** | Consultas por palabras clave exactas | Índice Lucene sobre el texto de los chunks |
| **Hybrid search** | Lo mejor de ambos mundos | Combina y reordena resultados vectoriales + full-text |
| **Text2Cypher** | Conteo, agregación, traversal del grafo | El LLM genera una consulta Cypher a partir de lenguaje natural |
| **Manual queries** | Preguntas frecuentes del dominio | Plantillas Cypher parametrizadas escritas a mano |

**Requisito previo:** Ejecutar `01_ingestion_demo.ipynb` primero para cargar los datos de Sherlock Holmes en Neo4j.

## 1. Configuración

In [1]:
import sys
sys.path.append('..')

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.retrieval import VectorRetriever, HybridRetriever, FullTextRetriever, Text2CypherRetriever, ManualRetriever
from graphrag.agents import AgenticRAG

neo4j = Neo4jManager()

# Verificar estado del grafo
stats = neo4j.get_stats()
print("Conectado a Neo4j.")
print(f"Estado del grafo: {stats}")

Conectado a Neo4j.
Estado del grafo: {'Event': 416, 'Deduction': 326, 'Object': 270, 'Chunk': 200, 'Scene': 181, 'Location': 88, 'Character': 73, 'Crime': 19, 'Story': 6}


## 2. Vector search — similitud semántica

La consulta se embede con el mismo modelo usado en la ingestión (`nomic-embed-text`).
Neo4j devuelve los `top_k` chunks cuyos vectores de embedding son más cercanos al vector de la consulta.

Ideal para: *"¿Cómo resuelve Holmes el caso?"*, *"Describe la relación entre Holmes y Watson"*

In [2]:
vector_retriever = VectorRetriever(neo4j)

results = vector_retriever.retrieve("How does Holmes solve the case in A Scandal in Bohemia?")

print(f"Vector search — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f} | story={r.get('story_title', 'N/A')}")
    print(f"   {r['text'][:180]}...\n")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n        YIELD node AS chunk, score\n        MATCH (s:Story)-[:HAS_CHUNK]->(chunk)\n        \n        RETURN chunk.text AS text,\n               chunk.id AS chunk_id,\n               chunk.chunk_index AS chunk_index,\n               chunk.position AS posi

Vector search — 10 resultados

1. score=0.875 | story=A Scandal In Bohemia
   her instinct is at once to rush to the thing which she values most. It is a perfectly overpowering impulse, and I have more than once taken advantage of it. In the case of the Darl...

2. score=0.857 | story=A Scandal In Bohemia
   "Because it would spare your Majesty all fear of future annoyance. If the lady loves her husband, she does not love your Majesty. If she does not love your Majesty, there is no rea...

3. score=0.852 | story=A Scandal In Bohemia
   alarm. Slipping through the shouting crowd I made my way to the corner of the street, and in ten minutes was rejoiced to find my friend's arm in mine, and to get away from the scen...

4. score=0.848 | story=The Adventure Of The Dancing Men
   always a look of fear upon her face—a look as if she were waiting and expecting. She would do better to trust me. She would find that I was her best friend. But until she speaks, I...

5. score=0.845 | story=A Scan

In [3]:
results = vector_retriever.retrieve_with_context("What deductions does Holmes make about the visitor?")

print(f"Vector search con contexto — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f} | story={r.get('story_title', 'N/A')}")
    print(f"   Personajes: {[c['name'] for c in r.get('characters', [])]}")
    print(f"   Ubicaciones: {[l['name'] for l in r.get('locations', [])]}")
    print(f"   {r['text'][:150]}...\n")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n        YIELD node AS chunk, score\n        MATCH (s:Story)-[:HAS_CHUNK]->(chunk)\n        \n        OPTIONAL MATCH (chunk)-[:MENTIONS]->(c:Character)\n        WITH chunk, score, s, [x IN collect(DISTINCT c) WHERE x IS NOT NULL | {name: x.name, occupatio

Vector search con contexto — 10 resultados

1. score=0.867 | story=The Red-Headed League
   Personajes: ['Sherlock Holmes']
   Ubicaciones: []
   save his blazing red head, and the expression of extreme chagrin and discontent upon his features. Sherlock Holmes' quick eye took in my occupation, a...

2. score=0.839 | story=The Final Problem
   Personajes: ['Watson', 'Peter Steiler the elder']
   Ubicaciones: []
   of a man who sees the fulfillment of that which he had expected. And yet for all his watchfulness he was never depressed. On the contrary, I can never...

3. score=0.836 | story=Silver Blaze
   Personajes: ['Watson', 'John Straker', 'Colonel Ross']
   Ubicaciones: []
   "You have formed a theory, then?" "At least I have got a grip of the essential facts of the case. I shall enumerate them to you, for nothing clears up...

4. score=0.833 | story=A Scandal In Bohemia
   Personajes: []
   Ubicaciones: []
   matters which are of an importance which can hardly be exaggerated. This 

## 3. Full-text search — coincidencia por palabras clave

Utiliza un índice Lucene de texto completo sobre el texto de los chunks.
Rápido y preciso para términos específicos, pero no detecta paráfrasis.

Ideal para: *"Baker Street"*, *"Irene Adler"*, *"strychnine"*

In [4]:
fulltext_retriever = FullTextRetriever(neo4j)

results = fulltext_retriever.retrieve("Baker Street")

print(f"Full-text search — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f} | story={r.get('story_title', 'N/A')}")
    print(f"   {r['text'][:180]}...\n")

Full-text search — 10 resultados

1. score=3.096 | story=A Scandal In Bohemia
   a satisfaction to his Majesty to regain it with his own hands." "And when will you call?" "At eight in the morning. She will not be up, so that we shall have a clear field. Besides...

2. score=2.935 | story=A Scandal In Bohemia
   interests which rise up around the man who first finds himself master of his own establishment, were sufficient to absorb all my attention, while Holmes, who loathed every form of ...

3. score=2.935 | story=The Final Problem
   his absence might mean that some blow had fallen during the night. Already the doors had all been shut and the whistle blown, when— "My dear Watson," said a voice, "you have not ev...

4. score=2.391 | story=A Scandal In Bohemia
   Holmes changed his costume. His expression, his manner, his very soul seemed to vary with every fresh part that he assumed. The stage lost a fine actor, even as science lost an acu...

5. score=2.391 | story=The Red-Headed Lea

## 4. Hybrid search — combinado y reordenado

Ejecuta búsqueda vectorial y de texto completo en paralelo, normaliza las puntuaciones dentro de cada rama,
luego hace la unión y reordena por la puntuación normalizada máxima.

Generalmente la estrategia más robusta para consultas abiertas.

In [5]:
hybrid_retriever = HybridRetriever(neo4j)

results = hybrid_retriever.retrieve("Holmes disguise King of Bohemia")

print(f"Hybrid search — {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r['score']:.3f} | story={r.get('story_title', 'N/A')}")
    print(f"   {r['text'][:180]}...\n")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=3, column=5, offset=15>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 15, 'line': 3, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL () {\n    CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n    YIELD node, score\n    WITH collect({node: node, score: score}) AS nodes, max(score) AS max_score\n    UNWIND nodes AS n\n    RETURN n.node AS node,\n           CASE WHEN max_score > 0 THEN n.score / max_score ELSE 0 END AS score\n    UNION\

Hybrid search — 10 resultados

1. score=1.000 | story=A Scandal In Bohemia
   The King stared at him in amazement. "Irene's photograph!" he cried. "Certainly, if you wish it." "I thank your Majesty. Then there is no more to be done in the matter. I have the ...

2. score=1.000 | story=A Scandal In Bohemia
   Sit down in that armchair, Doctor, and give us your best attention." A slow and heavy step, which had been heard upon the stairs and in the passage, paused immediately outside the ...

3. score=0.994 | story=A Scandal In Bohemia
   vanished into the bedroom, whence he emerged in five minutes tweed-suited and respectable, as of old. Putting his hands into his pockets, he stretched out his legs in front of the ...

4. score=0.993 | story=A Scandal In Bohemia
   seriously compromise one of the reigning families of Europe. To speak plainly, the matter implicates the great House of Ormstein, hereditary kings of Bohemia." "I was also aware of...

5. score=0.990 | story=The Adventure Of T

## 5. Text2Cypher — lenguaje natural a consulta de grafo

Un LLM (Gemini Pro) traduce la pregunta en lenguaje natural a una consulta Cypher,
que se ejecuta directamente contra Neo4j.

Imprescindible para preguntas que requieren **conteo**, **agregación** o
**traversal multi-salto** del grafo — cosas que la búsqueda vectorial no puede responder.

In [6]:
text2cypher = Text2CypherRetriever(neo4j)

In [7]:
# Conteo simple
cypher, results = text2cypher.retrieve("How many characters appear in A Scandal in Bohemia?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")

Cypher generado:
  MATCH (c:Character)-[:APPEARS_IN]->(s:Story) WHERE toLower(s.title) = toLower('A Scandal in Bohemia') RETURN count(c) AS number_of_characters

Resultados:
  {'number_of_characters': 13}


In [8]:
# Traversal de relaciones
cypher, results = text2cypher.retrieve("What crimes does Sherlock Holmes investigate?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")

Cypher generado:
  MATCH (c:Character {name: 'Sherlock Holmes'})-[:INVESTIGATES]->(cr:Crime) RETURN cr.name AS crime, cr.type AS type, cr.description AS description, cr.story_title AS story

Resultados:
  {'crime': 'Disappearance of the favourite for the Wessex Cup', 'type': 'disappearance', 'description': "Colonel Ross's horse, Silver Blaze, is still missing | John Straker is believed to have taken out Silver Blaze from the stables at night. | John Straker planned to make a slight, untraceable nick on the tendons of Silver Blaze's ham using a surgical knife to cause lameness, which necessitated taking the horse out onto the moor. | Silas Brown hid the favourite horse, Silver Blaze, to prevent it from winning against his own bet. | The disappearance of Colonel Ross's horse, Silver Blaze. | The disappearance of Colonel Ross's horse, which Holmes is tasked with recovering. | The disappearance of Silver Blaze, the racehorse. | The disappearance of the favourite horse for the Wessex Cup. |

In [9]:
# Agregación
cypher, results = text2cypher.retrieve("How many stories are there per collection?")

print("Cypher generado:")
print(f"  {cypher}\n")
print("Resultados:")
for r in results:
    print(f"  {r}")

Cypher generado:
  MATCH (s:Story) RETURN s.collection AS collection, count(s) AS number_of_stories ORDER BY number_of_stories DESC

Resultados:
  {'collection': 'The Adventures of Sherlock Holmes', 'number_of_stories': 3}
  {'collection': 'The Memoirs of Sherlock Holmes', 'number_of_stories': 2}
  {'collection': 'The Return of Sherlock Holmes', 'number_of_stories': 1}


## 6. Manual queries — plantillas Cypher parametrizadas

Consultas Cypher escritas a mano para preguntas frecuentes del dominio.
Ofrecen rendimiento garantizado y resultados precisos.

In [10]:
manual = ManualRetriever(neo4j)

# Ver plantillas disponibles
print("Plantillas disponibles:")
for q in manual.get_available_queries():
    print(f"  - {q['name']}: {q['description']}")

print("\n" + "="*60 + "\n")

# Ejecutar una plantilla
cypher, results = manual.retrieve("characters_in_story", {"story_title": "A Scandal in Bohemia"})
print("Personajes en 'A Scandal in Bohemia':")
for r in results:
    print(f"  - {r}")

Plantillas disponibles:
  - characters_in_story: Lista todos los personajes que aparecen en un relato específico
  - stories_of_character: Lista todos los relatos en los que aparece un personaje
  - recurring_characters: Personajes que aparecen en un número mínimo de relatos
  - crimes_investigated_by: Crímenes que investiga un personaje específico
  - character_relationships_in_story: Pares de personajes que se conocen entre sí dentro de un relato
  - locations_in_story: Ubicaciones donde transcurren las escenas de un relato
  - deduction_chain: Cadena de deducciones de Holmes en un relato, con sus encadenamientos
  - objects_used_by: Objetos que utiliza un personaje
  - story_summary: Resumen estadístico de un relato: personajes, crímenes, escenas y deducciones
  - crimes_by_type: Agrupa todos los crímenes del corpus por tipo con su recuento


Personajes en 'A Scandal in Bohemia':
  - {'name': 'Atkinson brothers', 'occupation': '', 'description': 'Victims of a singular tragedy that H

In [11]:
# Personajes recurrentes
cypher, results = manual.retrieve("recurring_characters", {"min_stories": 2})
print("Personajes que aparecen en 2+ relatos:")
for r in results:
    print(f"  - {r}")

Personajes que aparecen en 2+ relatos:
  - {'name': 'Sherlock Holmes', 'occupation': 'Consulting Detective', 'story_count': 6}
  - {'name': 'Watson', 'occupation': 'doctor', 'story_count': 6}
  - {'name': 'Mrs. Hudson', 'occupation': 'housekeeper', 'story_count': 2}


## 7. RAG Agéntico — orquestación automática

El sistema `AgenticRAG` integra todos los retrievers y selecciona automáticamente
el más adecuado para cada pregunta. El flujo es:

1. **Router** analiza la pregunta y elige la herramienta
2. **Retriever** recupera contexto relevante
3. **LLM** genera la respuesta basada en el contexto
4. **Critic** evalúa si la respuesta es completa y fiel
5. Si no, se reintenta con sub-preguntas refinadas

In [12]:
rag = AgenticRAG(neo4j)

# Batería de preguntas que deberían triggerear distintos retrievers
questions = [
    "Hello, what can you do?",                                    # → greeting
    "What is the weather like today?",                             # → out_of_scope
    "How does Holmes solve the mystery in A Scandal in Bohemia?",  # → vector_search
    "How many characters appear in the graph?",                    # → text2cypher
    "Who does Sherlock Holmes know?",                              # → text2cypher
    "Describe Holmes' method of deduction",                        # → vector_search
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")

    result = rag.answer(q)

    # routing_decision es un dict en nuestra implementación (ya serializado con model_dump)
    last_iter = result["iterations"][-1]
    routing = last_iter.get("routing_decision", {})
    print(f"Router → {routing.get('tool', 'N/A')}")
    if routing.get("reasoning"):
        print(f"Razón: {routing['reasoning'][:80]}")

    print(f"Respuesta: {result['answer']}")
    print(f"Iteraciones: {result.get('total_iterations', len(result['iterations']))}")


Q: Hello, what can you do?
Router → skills
Razón: The user is explicitly asking about the system's capabilities with the phrase 'w
Respuesta: Puedo responder preguntas sobre los relatos de Sherlock Holmes: personajes y sus relaciones, ubicaciones, crímenes e investigaciones, deducciones y métodos de razonamiento de Holmes, objetos y pistas, escenas y eventos. Puedo buscar por similitud semántica, por palabras clave exactas, o consultar directamente el grafo de conocimiento para preguntas estructuradas.
Iteraciones: 1

Q: What is the weather like today?
Router → out_of_scope
Razón: The user is asking about the current weather, which is a topic unrelated to the 
Respuesta: Esta pregunta está fuera de mi ámbito. Solo puedo responder preguntas relacionadas con las historias de Sherlock Holmes y su universo literario.
Iteraciones: 1

Q: How does Holmes solve the mystery in A Scandal in Bohemia?


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n        YIELD node AS chunk, score\n        MATCH (s:Story)-[:HAS_CHUNK]->(chunk)\n        \n        RETURN chunk.text AS text,\n               chunk.id AS chunk_id,\n               chunk.chunk_index AS chunk_index,\n               chunk.position AS posi

Router → vector_search
Razón: The user is asking a descriptive 'how' question about the resolution of a specif
Respuesta: Holmes solves the mystery by staging a false alarm to make the woman reveal the hiding place of the photograph [1, 2]. He then returns to the house, finds the lady gone, and retrieves the photograph and a letter from a recess behind a sliding panel [3].
Iteraciones: 1

Q: How many characters appear in the graph?
Router → text2cypher
Razón: The user is asking for a specific count ('How many') of a particular entity type
Respuesta: 73 [1].
Iteraciones: 1

Q: Who does Sherlock Holmes know?
Router → text2cypher
Razón: The user is asking for a list of characters who have a relationship ('knows') wi
Respuesta: Sherlock Holmes conoce a Abe Slaney [1], Colonel [2], Count Von Kramm [3], Dawson [4], Gregory [5], Irene Adler [6], Jabez Wilson [7], John Clay [8], Master Silas Brown [9], Miss Roylott [10], Mr. Merryweather [11], Mrs. Farintosh [12], Mrs. Hudson [13], Mrs. King [

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n        YIELD node AS chunk, score\n        MATCH (s:Story)-[:HAS_CHUNK]->(chunk)\n        \n        RETURN chunk.text AS text,\n               chunk.id AS chunk_id,\n               chunk.chunk_index AS chunk_index,\n               chunk.position AS posi

Router → vector_search
Razón: The user is asking for a description of a concept (Holmes' method of deduction).
Respuesta: Holmes believes it is a capital mistake to theorize before one has data, as one might twist facts to suit theories [1]. His method involves detaching the framework of absolute undeniable fact from embellishments, establishing a sound basis, and then drawing inferences to identify the special points upon which the mystery turns [4]. He constructs a series of inferences, each dependent upon its predecessor and simple in itself, which can produce a startling effect when only the starting-point and conclusion are presented [2, 5].
Iteraciones: 1


In [13]:
# Demostrar contexto conversacional multi-turno
rag.reset_conversation()

result1 = rag.answer("What happens in A Scandal in Bohemia?")
print(f"Q1: What happens in A Scandal in Bohemia?")
print(f"A1: {result1['answer']}\n")

result2 = rag.answer("Who are the main characters in that story?")
print(f"Q2: Who are the main characters in that story?")
print(f"A2: {result2['answer']}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n        CALL db.index.vector.queryNodes('chunk_embeddings', $top_k, $query_embedding)\n        YIELD node AS chunk, score\n        MATCH (s:Story)-[:HAS_CHUNK]->(chunk)\n        \n        RETURN chunk.text AS text,\n               chunk.id AS chunk_id,\n               chunk.chunk_index AS chunk_index,\n               chunk.position AS posi

Q1: What happens in A Scandal in Bohemia?
A1: Sherlock Holmes intentó recuperar una fotografía de Irene Adler disfrazándose de clérigo y orquestando una falsa alarma de incendio en su casa para que ella revelara dónde la guardaba [1, 3, 5, 6]. El resultado fue que Holmes descubrió el escondite de la fotografía, pero Irene Adler se dio cuenta de su engaño, huyó con su marido y se llevó la fotografía, dejando una carta para Holmes [2, 5]. Irene Adler superó a Sherlock Holmes porque había sido advertida sobre él, se dio cuenta de su estratagema después de la alarma de incendio, y, usando su experiencia como actriz y su habilidad para disfrazarse, lo siguió para confirmar sus sospechas antes de huir con la fotografía [2].

Q2: Who are the main characters in that story?
A2: Los personajes principales son Sherlock Holmes, Watson, Irene Adler y el Rey de Bohemia [6, 7, 11, 12].


In [14]:
neo4j.close()